# Inference Demo - German to Englisch

* Encoder-decoder Transformer with 48.2M parameters

* BPE tokenizer implemented and trained from scratch

* Trained on the Multi30k dataset

In [1]:
import torch

from neuralmt import (
    Multi30kDataset,
    BPETokenizer,
    NeuralMachineTranslator,
)

In [2]:
# Tokenizer file and nmt weights
TOKENIZER_FILE = "tokenizer_de_en_bpe_8k.pickle"
NMT_WEIGHTS_FILE = "nmt-de2en-48.2M-params.pt"

TOKENIZER = BPETokenizer.load(TOKENIZER_FILE)
LANGUAGE_PAIR = ("de", "en")    # Englisch to german
PAD_TOKEN = TOKENIZER.pad_id    # 7997
BOS_TOKEN = TOKENIZER.bos_id    # 7998
EOS_TOKEN = TOKENIZER.eos_id    # 7999
PAD_STRING = "<PAD>"
BOS_STRING = "<BOS>"
EOS_STRING = "<EOS>"

# Hyperparameters Transformer
# Some transformer hyperparamters from:
# https://arxiv.org/abs/1706.03762
# NOTE: These hyperparameters max out my RTX 3060 TI (8gb VRAM)
VOCAB_SIZE = len(TOKENIZER)     # 8000
MAX_SEQ_LENGTH = 256
D_MODEL = 512
NUM_LAYERS = 6
NUM_HEADS = 8
DK_DIM = 64
DV_DIM = 64
FF_DIM = 1365
P_DROPOUT = 0.5

nmt = NeuralMachineTranslator(
    NUM_LAYERS,
    NUM_HEADS,
    D_MODEL,
    VOCAB_SIZE,
    MAX_SEQ_LENGTH,
    FF_DIM,
    DK_DIM,
    DV_DIM,
    P_DROPOUT,
)
nmt.load_state_dict(torch.load(NMT_WEIGHTS_FILE, map_location="cpu"))

<All keys matched successfully>

In [3]:
nmt.eval()

NeuralMachineTranslator(
  (embedding): Embedding(8000, 512)
  (embedding_dropout): Dropout(p=0.5, inplace=False)
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (norm_1): RMSNorm((512,), eps=None, elementwise_affine=True)
        (mhd): MultiHeadAttention(
          (q_layer): Linear(in_features=512, out_features=512, bias=False)
          (k_layer): Linear(in_features=512, out_features=512, bias=False)
          (v_layer): Linear(in_features=512, out_features=512, bias=False)
          (out_proj): Linear(in_features=512, out_features=512, bias=False)
          (dropout): Dropout(p=0.5, inplace=False)
          (rope): RoPE()
        )
        (dropout_1): Dropout(p=0.5, inplace=False)
        (norm_2): RMSNorm((512,), eps=None, elementwise_affine=True)
        (ffn): FFNSwiGLU(
          (gate): Linear(in_features=512, out_features=1365, bias=True)
          (down): Linear(in_features=512, out_features=1365, bias=True)
          (up): Linear(in_

In [4]:
# Creating a prompt
prompt = "Leute, die vor einem Gebäude stehen."
tokens = [BOS_TOKEN] + TOKENIZER.encode(prompt) + [EOS_TOKEN]
print(
    f"Prompt: {prompt}\n"
    f"Tokens: {tokens}"
)

Prompt: Leute, die vor einem Gebäude stehen.
Tokens: [7998, 1610, 44, 393, 439, 293, 1068, 689, 46, 7999]


In [5]:
# Translating the prompt from englisch to german
tokens_tensor = torch.as_tensor(tokens, dtype=torch.int64).unsqueeze(0)
tokens_output_tensor = nmt.generate(tokens_tensor, BOS_TOKEN, EOS_TOKEN, PAD_TOKEN)
tokens_output = tokens_output_tensor.detach().cpu().squeeze(0).tolist()
output = TOKENIZER.decode(tokens_output, skip_special_tokens=True)
print(
    f"Prompt:           {prompt}\n"
    f"Tokens (input):   {tokens}\n"
    f"Tokens (output):  {tokens_output}\n"
    f"Translation:      {output}"
)

Prompt:           Leute, die vor einem Gebäude stehen.
Tokens (input):   [7998, 1610, 44, 393, 439, 293, 1068, 689, 46, 7999]
Tokens (output):  [7998, 886, 531, 270, 587, 327, 257, 812, 46, 7999]
Translation:      People standing in front of a building.


In [6]:
test_set = Multi30kDataset(TOKENIZER, LANGUAGE_PAIR, "test")

n_samples = 32
for i in range(n_samples):
    sample = test_set[i]
    src = sample["src"]
    pred = nmt.generate(src.unsqueeze(0), BOS_TOKEN, EOS_TOKEN, PAD_TOKEN).squeeze(0)
    src_text = TOKENIZER.decode(src.tolist(), True)
    pred_text = TOKENIZER.decode(pred.detach().tolist(), True)    

    print(f"Query       ({LANGUAGE_PAIR[0]}):   {src_text}\n"
          f"Translation ({LANGUAGE_PAIR[1]}):   {pred_text}\n"
    )

Query       (de):   Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Translation (en):   A man in an orange hat looking at something from the right.

Query       (de):   Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
Translation (en):   A body of drill runs across a white fence.

Query       (de):   Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
Translation (en):   A girl in a karate uniform is plowing a board with a kick.

Query       (de):   Fünf Leute in Winterjacken und mit Helmen stehen im Schnee mit Schneemobilen im Hintergrund.
Translation (en):   Five people in winter jackets and helmets stand in the snow with snow on it.

Query       (de):   Leute Reparieren das Dach eines Hauses.
Translation (en):   People are using a roller coaster to a house.

Query       (de):   Ein hell gekleideter Mann fotografiert eine Gruppe von Männern in dunklen Anzügen und mit Hüten, die um eine Frau in einem trägerlosen Kleid herum stehen.
Tran